# Tratamento de Valores Ausentes
## Estratégias de Imputação e Análise

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 05
---

# Parte I: Conceitos Teóricos

# ==================================================
# 1. INTRODUÇÃO AOS VALORES AUSENTES
# ==================================================

OBJETIVOS DA AULA:
- Compreender o impacto de valores ausentes na análise de dados
- Identificar diferentes tipos de valores ausentes
- Aprender métodos de visualização e quantificação de dados faltantes
- Implementar técnicas para tratamento de valores ausentes
- Avaliar o impacto das diferentes estratégias de imputação

In [ ]:
# @title
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

# ==================================================
# 2. IMPORTÂNCIA DA QUALIDADE DOS DADOS
# ==================================================

POR QUE A QUALIDADE DOS DADOS É CRUCIAL?

1. Decisões baseadas em dados imprecisos levam a resultados incorretos:
   - Análises enviesadas
   - Modelos de baixo desempenho
   - Conclusões errôneas

2. Estatísticas sobre problemas de qualidade:
   - Estima-se que cientistas de dados gastam 60-80% do tempo em limpeza de dados
   - Erros de dados custam às empresas 15-25% da receita
   - 80% dos projetos de análise falham devido à má qualidade dos dados

3. Impactos dos valores ausentes:
   - Redução do poder estatístico
   - Introdução de viés na análise
   - Muitos algoritmos não conseguem processar valores ausentes diretamente
   - Interpretação incorreta de relações entre variáveis
   
4. Garbage In, Garbage Out (GIGO):
   - Independente da sofisticação do modelo, dados ruins produzem resultados ruins
   - A qualidade da saída é diretamente proporcional à qualidade da entrada

# ==================================================
# 3. CRIANDO UM CONJUNTO DE DADOS COM VALORES AUSENTES
# ==================================================

In [ ]:
# @title

# Definindo a semente para reprodutibilidade
np.random.seed(42)
n = 1000

# Criando variáveis base
idade_cliente = np.random.normal(35, 12, n)
tempo_site = np.random.gamma(2, 15, n)
paginas_visitadas = np.random.poisson(8, n)
total_compras_anteriores = np.random.poisson(4, n)
tempo_cliente = np.random.gamma(5, 60, n)

# Criando relacionamentos mais fortes entre as variáveis
# Definindo o valor_compra com base em outras variáveis
base_valor = 20 + 0.5 * idade_cliente + 2 * np.log1p(tempo_site) + 5 * np.log1p(paginas_visitadas) + 10 * np.log1p(total_compras_anteriores)
# Adicionando ruído ao valor para torná-lo mais realista
valor_compra = base_valor + np.random.normal(0, 20, n)

# Calculando outras variáveis com base em relações mais realistas
avaliacao_produto = np.clip(np.round(3 + 0.5 * np.random.normal(0, 1, n) + 0.3 * (valor_compra / 100)), 1, 5)
tempo_entrega = np.clip(np.random.gamma(2, 3, n) - 0.01 * valor_compra + 0.5 * np.random.normal(0, 1, n), 1, 30)

# Probabilidade de devolução baseada em outros fatores
prob_devolucao = 1 / (1 + np.exp(2 - 0.5 * (5 - avaliacao_produto) + 0.01 * tempo_entrega - 0.005 * valor_compra))
devolveu_produto = np.random.binomial(1, prob_devolucao)

# Criando o DataFrame
data = {
    'idade_cliente': idade_cliente,
    'tempo_site_minutos': tempo_site,
    'paginas_visitadas': paginas_visitadas,
    'valor_compra': valor_compra,
    'avaliacao_produto': avaliacao_produto,
    'tempo_cliente_dias': tempo_cliente,
    'total_compras_anteriores': total_compras_anteriores,
    'tempo_entrega_dias': tempo_entrega,
    'devolveu_produto': devolveu_produto
}

df = pd.DataFrame(data)

In [ ]:
# @title
# Introduzindo valores ausentes no dataset com diferentes padrões
# 1. MCAR (Missing Completely At Random)
mcar_indices = np.random.choice(n, size=int(0.1 * n), replace=False)
df.loc[mcar_indices, 'idade_cliente'] = np.nan

# 2. MAR (Missing At Random) - Relacionado a outras variáveis
# Clientes mais jovens tendem a não informar o tempo que são clientes
young_indices = df[df['idade_cliente'] < 25].index
mar_indices = np.random.choice(young_indices, size=int(0.3 * len(young_indices)), replace=False)
df.loc[mar_indices, 'tempo_cliente_dias'] = np.nan

# 3. MNAR (Missing Not At Random) - Relacionado ao próprio valor
# Clientes com compras de alto valor tendem a não avaliar o produto
high_value_indices = df[df['valor_compra'] > df['valor_compra'].quantile(0.8)].index
mnar_indices = np.random.choice(high_value_indices, size=int(0.4 * len(high_value_indices)), replace=False)
df.loc[mnar_indices, 'avaliacao_produto'] = np.nan

# 4. Padrão Estrutural - Valores ausentes em conjunto
# Quando o cliente devolve o produto, muitas vezes não temos a avaliação e tempo de entrega
returned_indices = df[df['devolveu_produto'] == 1].index
for idx in returned_indices:
    if np.random.random() < 0.7:
        df.loc[idx, 'avaliacao_produto'] = np.nan
        df.loc[idx, 'tempo_entrega_dias'] = np.nan

# 5. Ausentes em múltiplas colunas
# Alguns registros têm várias informações ausentes
multi_missing_indices = np.random.choice(n, size=int(0.05 * n), replace=False)
for idx in multi_missing_indices:
    missing_cols = np.random.choice(
        ['paginas_visitadas', 'tempo_site_minutos', 'total_compras_anteriores'],
        size=np.random.randint(1, 4),
        replace=False
    )
    for col in missing_cols:
        df.loc[idx, col] = np.nan

In [ ]:
# @title
# Visualizando as primeiras linhas do DataFrame
print("Primeiras linhas do DataFrame com valores ausentes:")
df.head()

# ==================================================
# 4. TIPOS DE VALORES AUSENTES
# ==================================================

PRINCIPAIS CATEGORIAS DE VALORES AUSENTES:

1. MCAR (Missing Completely At Random)
   - Ausência não depende de nenhuma variável observada ou não observada
   - Como um sorteio aleatório: cada valor tem a mesma probabilidade de estar ausente
   - Exemplo: Falha em equipamento de coleta, perda acidental de dados
   - Impacto: Reduz o poder estatístico, mas não introduz viés sistemático
   
2. MAR (Missing At Random)
   - Ausência relacionada a outras variáveis observadas
   - A probabilidade de um valor estar ausente depende de dados que temos
   - Exemplo: Homens tendem a não responder questões sobre saúde mental
   - Impacto: Pode introduzir viés se não considerado nas análises

3. MNAR (Missing Not At Random)
   - Ausência relacionada ao próprio valor ausente
   - O valor em si influencia sua probabilidade de estar ausente
   - Exemplo: Pessoas com renda alta tendem a não informar renda
   - Impacto: Introduz viés significativo, difícil de corrigir

4. Ausência Estrutural
   - Valores ausentes por design ou lógica do problema
   - Exemplo: Tempo de entrega ausente para compras não finalizadas
   - Impacto: Deve ser interpretado diferentemente de outros tipos de ausência

# ==================================================
# 5. IDENTIFICAÇÃO E VISUALIZAÇÃO DE VALORES AUSENTES
# ==================================================

In [ ]:
# @title
# Verificando a quantidade de valores ausentes por coluna
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

# Criando um DataFrame para visualização
missing_df = pd.DataFrame({
    'Total Ausentes': missing_values,
    'Percentual (%)': missing_percent
})

print("\nResumo de valores ausentes por coluna:")
print(missing_df[missing_df['Total Ausentes'] > 0].sort_values('Percentual (%)', ascending=False))

In [ ]:
# @title
# Visualizando o padrão de valores ausentes
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Padrão de Valores Ausentes',fontsize=14)
plt.xlabel('Variáveis')
plt.ylabel('Observações')
plt.show()

In [ ]:
# @title
# Usando a biblioteca missingno para visualização especializada
plt.figure(figsize=(12, 6))
msno.matrix(df)
plt.title('Matriz de Valores Ausentes (missingno)',fontsize=20)
plt.show()

In [ ]:
# @title
# Visualizando a correlação entre valores ausentes
plt.figure(figsize=(10, 8))
msno.heatmap(df)
plt.title(
    "Correlação entre Valores Ausentes\n"
    "Valores próximos de 1 indicam ausência simultânea",
    fontsize=18
)
plt.show()

In [ ]:
# @title
# Análise de valor ausente por variável
plt.figure(figsize=(12, 6))
msno.bar(df)
plt.title('Completude por Variável',fontsize=18)
plt.show()

# ==================================================
# 6. ANÁLISE EXPLORATÓRIA COM VALORES AUSENTES
# ==================================================

In [ ]:
# @title
# Analisando se valores ausentes estão relacionados a outras variáveis (MAR)
# Exemplo: idade_cliente vs. tempo_cliente_dias ausente

plt.figure(figsize=(12, 6))
# Boxplot comparando idades de quem informou tempo_cliente vs quem não informou
sns.boxplot(x=df['tempo_cliente_dias'].isnull(), y=df['idade_cliente'])
plt.title('Distribuição de Idade por Ausência em Tempo Cliente',fontsize=14)
plt.xlabel('Tempo Cliente (informado?)')
plt.ylabel('Idade do Cliente')
plt.xticks([0, 1], ['Informado', 'Não Informado'])
plt.show()

In [ ]:
# @title
# Analisando se valores ausentes estão relacionados ao próprio valor (MNAR)
# Exemplo: valor_compra vs. avaliacao_produto ausente

plt.figure(figsize=(12, 6))
sns.boxplot(x=df['avaliacao_produto'].isnull(), y=df['valor_compra'])
plt.title('Distribuição de Valor da Compra por Ausência em Avaliação',fontsize=14)
plt.xlabel('Avaliação Ausente')
plt.ylabel('Valor da Compra')
plt.xticks([0, 1], ['Avaliado', 'Não Avaliado'])
plt.show()

In [ ]:
# @title
# Comparando correlações entre variáveis considerando valores não ausentes
# Grupo com valores completos
df_complete = df.dropna(subset=['avaliacao_produto'])
corr_complete = df_complete[['valor_compra', 'avaliacao_produto', 'devolveu_produto']].corr()

# Visualizando a correlação
plt.figure(figsize=(10, 8))
sns.heatmap(corr_complete, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlação entre Variáveis (Apenas Registros Completos)',fontsize=14)
plt.show()

# ==================================================
# 7. COMO DETECTAR O TIPO DE AUSÊNCIA (MCAR, MAR, MNAR)
# ==================================================

MÉTODOS PARA DETECTAR O TIPO DE AUSÊNCIA:

1. Para testar MCAR (Missing Completely At Random):

   *(Ausência não depende de nenhuma variável observada ou não observada)*
   - Teste de Little: teste estatístico formal para MCAR
   - Comparação de distribuições entre grupos com valores ausentes e completos
   - Análise de correlação entre indicadores de ausência

2. Para identificar MAR (Missing At Random):

   *(Ausência relacionada a outras variáveis observadas)*
   - Analisar relações entre ausência em uma variável e valores de outras variáveis
   - Teste t/Wilcoxon entre grupos com valores presentes e ausentes
   - Modelar a probabilidade de ausência com outras variáveis

3. Para identificar MNAR (Missing Not At Random):

   *(Ausência relacionada ao próprio valor ausente)*
   - Mais difícil de detectar diretamente
   - Análise de sensibilidade
   - Informações contextuais e do domínio
   - Padrões temporais ou espaciais

4. Ferramentas práticas:
   - Visualizações (heatmaps, boxplots comparativos)
   - Testes estatísticos de independência
   - Análise de padrões de ausência em subgrupos

In [ ]:
# @title
# Exemplo de teste para MCAR vs. MAR: comparação de médias
from scipy import stats

# Comparando idade entre quem informou tempo_cliente e quem não informou
has_tempo = df[~df['tempo_cliente_dias'].isnull()]['idade_cliente'].dropna()
no_tempo = df[df['tempo_cliente_dias'].isnull()]['idade_cliente'].dropna()

# Teste t para comparar médias
t_stat, p_value = stats.ttest_ind(has_tempo, no_tempo, equal_var=False)
print(f"\nTeste t para comparar idade entre quem informou tempo_cliente e quem não:")
print(f"Estatística t: {t_stat:.4f}")
print(f"p-valor: {p_value:.4f}")
print(f"Conclusão: {'Provavelmente não é MCAR (é MAR)' if p_value < 0.05 else 'Pode ser MCAR'}")

In [ ]:
# @title
# Comparando valor_compra entre quem avaliou e quem não avaliou
rated = df[~df['avaliacao_produto'].isnull()]['valor_compra']
not_rated = df[df['avaliacao_produto'].isnull()]['valor_compra']

# Teste t para comparar médias
t_stat, p_value = stats.ttest_ind(rated, not_rated, equal_var=False)
print(f"\nTeste t para comparar valor_compra entre quem avaliou e quem não:")
print(f"Estatística t: {t_stat:.4f}")
print(f"p-valor: {p_value:.4f}")
print(f"Conclusão: {'Provavelmente MNAR' if p_value < 0.05 else 'Pode ser MCAR ou MAR'}")

## 🎯 Micro-atividade 1: Missing Values CSI - Investigação Guiada (10 min)

**Objetivo**: Usar técnicas de visualização especializada para descobrir padrões ocultos nos valores ausentes

**Cenário**: Você é um "detetive de dados" investigando padrões suspeitos de valores ausentes no dataset de e-commerce.

### Parte 1: Observe os padrões

🔍 **PISTA 1 - Dendrograma:**

  - Variáveis próximas no dendrograma tendem a ter valores ausentes juntas
  - Quanto menor a altura da ligação, mais forte a relação

❓ Sua observação: Quais variáveis parecem estar relacionadas?

In [ ]:
# @title
# 🕵️ Código de Investigação - Execute e observe!

# 1. Dendrograma para investigar agrupamentos
import missingno as msno
plt.figure(figsize=(12, 6))
msno.dendrogram(df)
plt.title('Dendrograma de Valores Ausentes - Quais variáveis "andam juntas"?', fontsize=18)
plt.show()


🔍 **PISTA 2 - Correlações:**

  - Correlação > 0.3: Variáveis tendem a estar ausentes JUNTAS
  - Correlação < -0.3: Quando uma está ausente, a outra tende a estar PRESENTE
  - Correlação ≈ 0: Ausências são independentes (possível MCAR)

❓ Sua observação: Encontrou alguma correlação suspeita?

In [ ]:
# @title


# 2. Matriz de correlação de ausências
missing_corr = df.isnull().corr()

plt.figure(figsize=(10, 8))
sns.heatmap(missing_corr, annot=True, cmap='RdBu_r', center=0,
           fmt='.2f', square=True, cbar_kws={'label': 'Correlação'})
plt.title('Mapa de Calor: Correlações entre Ausências\n(Valores próximos de ±1 são suspeitos!)', fontsize=14)
plt.show()


💡 **Nota sobre valor_compra no heatmap:**

Observe que `valor_compra` e `devolveu_produto` aparecem sem valores (em branco) no mapa de calor. Isso ocorre porque:
- Estas variáveis **não possuem valores ausentes** em nosso dataset
- A correlação de ausência só pode ser calculada quando há variação (presença E ausência)
- Sem valores ausentes, não há variação para correlacionar

**Insight importante**: Isso é comum em datasets reais - variáveis críticas como preço geralmente são campos obrigatórios e não apresentam valores ausentes. O heatmap nos ajuda a identificar rapidamente quais variáveis têm dados completos!

In [ ]:
# @title
# 3. Análise automatizada de padrões suspeitos
print("🔍 PISTA 3 - Análise Automatizada de Padrões Suspeitos:\n")
print("=" * 60)

suspeitos = []
for i, col1 in enumerate(missing_corr.columns):
    for col2 in missing_corr.columns[i+1:]:
        corr_val = missing_corr.loc[col1, col2]
        if abs(corr_val) > 0.3:  # Correlação suspeita
            suspeitos.append((col1, col2, corr_val))
            print(f"⚠️  ALERTA: {col1} × {col2}")
            print(f"   Correlação: {corr_val:.3f}")
            if corr_val > 0.3:
                print(f"   → Tendem a estar ausentes JUNTAS")
            else:
                print(f"   → Padrão INVERSO de ausência")
            print()

if not suspeitos:
    print("✅ Nenhum padrão suspeito encontrado (correlações < 0.3)")

print("=" * 60)

### Parte 2: Sua Investigação 🕵️‍♂️

Agora responda as perguntas abaixo baseado nas visualizações:

**1. Padrão mais suspeito:**
- Qual par de variáveis tem a correlação mais alta?
- O que isso sugere sobre o comportamento dos clientes?

**2. Hipótese de negócio:**
- Por que `avaliacao_produto` e `tempo_entrega_dias` estariam ausentes juntas?
- Pense: Em que situação um cliente não teria essas duas informações?

**3. Classificação dos tipos:**
- `idade_cliente`: Parece ser MCAR, MAR ou MNAR? Por quê?
- `avaliacao_produto`: Qual tipo você suspeita? Que evidência você tem?

**4. Descoberta extra (Bonus):**
- Rode `df[df['devolveu_produto']==1][['avaliacao_produto', 'tempo_entrega_dias']].isnull().sum()`
- O que isso revela sobre a relação entre devolução e valores ausentes?

In [ ]:
# @title
# 📝 Espaço para suas respostas e investigações adicionais

# Teste do bonus - relação entre devolução e valores ausentes
print("🔍 INVESTIGAÇÃO BONUS:")
print("Valores ausentes quando o produto foi devolvido:")
print(df[df['devolveu_produto']==1][['avaliacao_produto', 'tempo_entrega_dias']].isnull().sum())
print("\nValores ausentes quando o produto NÃO foi devolvido:")
print(df[df['devolveu_produto']==0][['avaliacao_produto', 'tempo_entrega_dias']].isnull().sum())

# Seu código de investigação adicional aqui:
# Por exemplo:
# - Verificar se clientes mais jovens tendem a não informar certas variáveis
# - Analisar se valores de compra altos estão relacionados com ausências
# - Investigar padrões temporais de ausência

### Parte 3: Conclusões da Investigação 🎯

**Revelações importantes:**
1. **Padrão Estrutural**: Quando produtos são devolvidos, frequentemente não temos avaliação nem tempo de entrega
2. **MNAR Identificado**: Clientes com compras de alto valor tendem a não avaliar (correlação com `valor_compra`)
3. **MAR Confirmado**: Idade influencia se o cliente informa quanto tempo é cliente
4. **MCAR Validado**: `idade_cliente` tem ausências aleatórias (baixa correlação com outras)

**💡 Insight de negócio**:
Os padrões de ausência revelam comportamentos dos clientes:
- Devoluções impedem coleta de feedback
- Clientes premium podem ser menos engajados com avaliações
- Dados demográficos (idade) afetam disposição para compartilhar informações

**🚀 Próximo passo**:
Com esses padrões identificados, podemos escolher estratégias de imputação apropriadas para cada tipo!

# ==================================================
# 8. MÉTODOS DE TRATAMENTO DE VALORES AUSENTES
# ==================================================

ESTRATÉGIAS PARA LIDAR COM VALORES AUSENTES:

1. Remoção de Registros ou Variáveis/Colunas
   - Eliminação listwise (remover linhas com qualquer valor ausente)
   - Eliminação pairwise (usar todos os dados disponíveis para cada cálculo)
   - Remoção de variáveis/colunas com muitos dados ausentes

2. Métodos de Imputação Simples
   - Média, mediana, moda
   - Valor constante (zero, valor mínimo, etc.)
   - Forward/Backward fill (para dados temporais)

3. Métodos de Imputação Avançados
   - KNN (K-Nearest Neighbors)
   - Métodos baseados em regressão
   - Imputação múltipla
   - Algoritmos específicos (MICE, MissForest)

4. Indicadores de Ausência
   - Criação de variáveis dummy para indicar ausência
   - Combinação com métodos de imputação

5. Modelos que Lidam com Valores Ausentes
   - Árvores de decisão e métodos baseados em árvores
   - Extensões específicas de algoritmos

# ==================================================
# 9. REMOÇÃO DE REGISTROS OU VARIÁVEIS
# ==================================================

In [ ]:
# @title Eliminação listwise (remover todas as linhas com pelo menos um NA)
# Contando registros originais
print(f"Número original de registros: {len(df)}")

# Eliminação listwise (remover todas as linhas com pelo menos um NA)
df_dropna = df.dropna()
print(f"Número de registros após dropna(): {len(df_dropna)}")
print(f"Perda percentual: {(1 - len(df_dropna)/len(df))*100:.2f}%")

# Eliminação seletiva (apenas linhas com NAs em colunas específicas)
df_dropna_selective = df.dropna(subset=['idade_cliente', 'valor_compra'])
print(f"Número de registros após dropna seletivo ('idade_cliente', 'valor_compra'): {len(df_dropna_selective)}")
print(f"Perda percentual: {(1 - len(df_dropna_selective)/len(df))*100:.2f}%")

In [ ]:
# @title Eliminação pairwise (usar todos os dados disponíveis para cada cálculo)
# Contando registros originais
print(f"Número original de registros: {len(df)}\n")
# Aumenta a largura máxima do display
pd.set_option('display.width', 1000)

# Função para contar pares válidos (não nulos)
def contar_pairwise(df):
    cols = df.columns
    resultados = []
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            col1, col2 = cols[i], cols[j]
            # Conta apenas linhas sem NaN nas duas colunas
            n_validos = df[[col1, col2]].dropna().shape[0]
            perda = (1 - n_validos/len(df)) * 100
            resultados.append([col1, col2, n_validos, perda])
    return pd.DataFrame(resultados, columns=["Coluna 1", "Coluna 2", "Registros usados", "Perda (%)"])

# Aplicando no seu dataset
pairwise_stats = contar_pairwise(df)
print(pairwise_stats)


In [ ]:
# @title Removendo variáveis (colunas) com alta proporção de valores ausentes
# Removendo variáveis (colunas) com alta proporção de valores ausentes
threshold = 0.20  # 20% de valores ausentes
high_missing_cols = [col for col in df.columns if df[col].isnull().mean() > threshold]
print(f"\nVariáveis/Colunas com mais de {threshold*100}% de valores ausentes:")
print(high_missing_cols)

df_drop_cols = df.drop(columns=high_missing_cols)
print(f"Número de variáveis/colunas após remoção: {df_drop_cols.shape[1]} (original: {df.shape[1]})")

# ==================================================
# 10. IMPUTAÇÃO COM MÉTODOS SIMPLES
# ==================================================

In [ ]:
# @title
# Visualizando as distribuições originais para comparação
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols_with_na = [col for col in numeric_cols if df[col].isnull().sum() > 0]

# Criando uma cópia do DataFrame para imputação
df_imputed = df.copy()

# 1. Imputação pela média
imputer_mean = SimpleImputer(strategy='mean')
df_imputed[numeric_cols_with_na] = imputer_mean.fit_transform(df[numeric_cols_with_na])

# 2. Imputação pela mediana
df_median = df.copy()
imputer_median = SimpleImputer(strategy='median')
df_median[numeric_cols_with_na] = imputer_median.fit_transform(df[numeric_cols_with_na])

# 3. Imputação por valor constante (zero)
df_zero = df.copy()
imputer_zero = SimpleImputer(strategy='constant', fill_value=0)
df_zero[numeric_cols_with_na] = imputer_zero.fit_transform(df[numeric_cols_with_na])

# 4. Imputação para variáveis categóricas (exemplo com moda)
categorical_cols_with_na = ['avaliacao_produto']
df_imputed[categorical_cols_with_na] = SimpleImputer(strategy='most_frequent').fit_transform(df[categorical_cols_with_na])

In [ ]:
# @title
# Comparando as distribuições após imputação
for col in numeric_cols_with_na[:3]:  # Limitando a 3 colunas para melhor visualização
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 3, 1)
    sns.histplot(df[col].dropna(), kde=True, color='blue')
    plt.title(f'Original (sem NA) - {col}')
    plt.ylim(0, 200)  # Define os limites do eixo y

    plt.subplot(1, 3, 2)
    sns.histplot(df_imputed[col], kde=True, color='red')
    plt.title(f'Imputação pela Média - {col}')
    plt.ylim(0, 200)  # Define os limites do eixo y

    plt.subplot(1, 3, 3)
    sns.histplot(df_median[col], kde=True, color='green')
    plt.title(f'Imputação pela Mediana - {col}')
    plt.ylim(0, 200)  # Define os limites do eixo y

    plt.tight_layout()
    plt.show()

## 🎯 Micro-atividade 2: Imputation Battle - Atividade Guiada (8 min)

**Objetivo**: Comparar diferentes estratégias de imputação e escolher a melhor

**Cenário**: Você precisa decidir qual método de imputação usar para a variável `tempo_cliente_dias` (MAR - Missing At Random)

### Parte 1: O código abaixo aplica 3 métodos de imputação

In [ ]:
# @title
# 🥊 Imputation Battle - Código Guiado

# Guardando os valores originais para comparação
tempo_original = df['tempo_cliente_dias'].copy()
indices_com_valor = ~tempo_original.isnull()

# Método 1: Imputação pela Média
tempo_media = tempo_original.copy()
tempo_media.fillna(tempo_original.mean(), inplace=True)

# Método 2: Imputação pela Mediana
tempo_mediana = tempo_original.copy()
tempo_mediana.fillna(tempo_original.median(), inplace=True)

# Método 3: Imputação baseada em correlação com idade_cliente
from sklearn.linear_model import LinearRegression

# Preparando dados para regressão
df_temp = df[['idade_cliente', 'tempo_cliente_dias']].dropna()
X_train = df_temp[['idade_cliente']]
y_train = df_temp['tempo_cliente_dias']

# Treinando modelo simples
model = LinearRegression()
model.fit(X_train, y_train)

# Prevendo valores ausentes
tempo_interpolado = tempo_original.copy()
missing_mask = tempo_original.isnull()
if missing_mask.sum() > 0:
    X_missing = df.loc[missing_mask, ['idade_cliente']].dropna()
    if len(X_missing) > 0:
        predicted_values = model.predict(X_missing)
        tempo_interpolado.loc[X_missing.index] = predicted_values

print("✅ Três métodos de imputação aplicados!")
print(f"Valores originais ausentes: {tempo_original.isnull().sum()}")
print(f"Após imputação - Média: {tempo_media.isnull().sum()}")
print(f"Após imputação - Mediana: {tempo_mediana.isnull().sum()}")
print(f"Após imputação - Regressão: {tempo_interpolado.isnull().sum()}, considerando 'idade_cliente', 'tempo_cliente_dias'")

In [ ]:
# @title
# 📊 Visualização das Distribuições

plt.figure(figsize=(15, 5))

# Subplot 1: Histogramas sobrepostos
plt.subplot(1, 3, 1)
plt.hist(tempo_original.dropna(), bins=30, alpha=0.5, label='Original', color='blue', density=True)
plt.hist(tempo_media, bins=30, alpha=0.5, label='Média', color='red', density=True)
plt.hist(tempo_mediana, bins=30, alpha=0.5, label='Mediana', color='green', density=True)
plt.hist(tempo_interpolado.dropna(), bins=30, alpha=0.5, label='Regressão', color='purple', density=True)
plt.xlabel('Tempo Cliente (dias)')
plt.ylabel('Densidade')
plt.title('Comparação das Distribuições')
plt.legend()

# Subplot 2: KDE plots
plt.subplot(1, 3, 2)
sns.kdeplot(tempo_original.dropna(), label='Original', color='blue')
sns.kdeplot(tempo_media, label='Média', color='red')
sns.kdeplot(tempo_mediana, label='Mediana', color='green')
sns.kdeplot(tempo_interpolado.dropna(), label='Regressão', color='purple')
plt.xlabel('Tempo Cliente (dias)')
plt.ylabel('Densidade')
plt.title('Curvas de Densidade (KDE)')
plt.legend()

# Subplot 3: Boxplots
plt.subplot(1, 3, 3)
data_to_plot = [tempo_original.dropna(), tempo_media, tempo_mediana, tempo_interpolado.dropna()]
plt.boxplot(data_to_plot, tick_labels=['Original', 'Média', 'Mediana', 'Regressão'])
plt.ylabel('Tempo Cliente (dias)')
plt.title('Comparação via Boxplot')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# @title
# 📈 Análise Quantitativa dos Métodos

from scipy import stats

# Calculando métricas de comparação
print("📊 ANÁLISE COMPARATIVA DOS MÉTODOS:\n")
print("=" * 60)

# Estatísticas descritivas
methods = {
    'Original': tempo_original.dropna(),
    'Média': tempo_media,
    'Mediana': tempo_mediana,
    'Regressão': tempo_interpolado.dropna()
}

stats_comparison = pd.DataFrame()
for name, data in methods.items():
    stats_comparison[name] = [
        data.mean(),
        data.median(),
        data.std(),
        data.min(),
        data.max()
    ]

stats_comparison.index = ['Média', 'Mediana', 'Desvio Padrão', 'Mínimo', 'Máximo']
print("\n📋 Estatísticas Descritivas:")
print(stats_comparison.round(2))

# Teste de Kolmogorov-Smirnov para comparar distribuições
print("\n🔬 Teste Kolmogorov-Smirnov (comparação com distribuição original):")
print("-" * 60)
original_values = tempo_original.dropna()

for method_name, method_data in [('Média', tempo_media),
                                  ('Mediana', tempo_mediana),
                                  ('Regressão', tempo_interpolado.dropna())]:
    ks_stat, p_value = stats.ks_2samp(original_values, method_data)
    print(f"{method_name:10} - KS stat: {ks_stat:.4f}, p-valor: {p_value:.4f}")
    if p_value > 0.05:
        print(f"           ✅ Distribuições similares (p > 0.05)")
    else:
        print(f"           ⚠️  Distribuições diferentes (p < 0.05)")

print("\n" + "=" * 60)

Os métodos de imputação por média, mediana e regressão preservaram as principais características da distribuição original. O teste de Kolmogorov-Smirnov confirmou que não há diferença estatisticamente significativa entre as distribuições.

### Parte 2: Análise e Decisão 🏆

**Perguntas para reflexão:**

1. **Preservação da distribuição**: Qual método manteve a distribuição mais próxima da original?

2. **Impacto nos extremos**: Algum método criou valores irreais (muito altos ou muito baixos)?

3. **Relação com outras variáveis**: O método de regressão preservou a relação idade-tempo_cliente?

**🎯 Vencedor**: Com base nas análises, o método de **regressão** tende a ser melhor para dados MAR, pois:
- Preserva a relação com outras variáveis
- Não concentra todos os valores imputados em um único ponto
- Mantém a variabilidade dos dados

**💡 Insight importante**:
- Para MCAR: média ou mediana funcionam bem
- Para MAR: métodos baseados em outras variáveis (como regressão) são melhores
- Para MNAR: métodos mais sofisticados são necessários

# ==================================================
# 11. MÉTODOS AVANÇADOS DE IMPUTAÇÃO
# ==================================================

In [ ]:
# @title Imputação com Random Forest: Prever idade_cliente com base em outras variáveis
# Imputação com Random Forest (para uma variável específica)
# Exemplo: Prever idade_cliente com base em outras variáveis
cols_for_rf = [col for col in numeric_cols if col != 'idade_cliente' and df[col].isnull().sum() < 100]
df_rf = df.copy()

# Separando registros com e sem idade_cliente
df_with_age = df_rf.dropna(subset=['idade_cliente'])
df_without_age = df_rf[df_rf['idade_cliente'].isnull()]

# Treinando o modelo de Random Forest
X_train = df_with_age[cols_for_rf]
y_train = df_with_age['idade_cliente']

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Prevendo os valores ausentes
if len(df_without_age) > 0:
    X_missing = df_without_age[cols_for_rf]
    predicted_ages = rf_model.predict(X_missing)

    # Atualizando o DataFrame
    df_rf.loc[df_rf['idade_cliente'].isnull(), 'idade_cliente'] = predicted_ages

In [ ]:
# @title
# Visualizando a distribuição após imputação com RF
plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
sns.histplot(df['idade_cliente'].dropna(), kde=True, color='blue')
plt.title(f'Original (sem NA) - idade_cliente')
plt.ylim(0, 200)  # Define os limites do eixo y

plt.subplot(1, 3, 2)
sns.histplot(df_imputed['idade_cliente'], kde=True, color='red')
plt.title(f'Imputação pela Média - idade_cliente')
plt.ylim(0, 200)  # Define os limites do eixo y

plt.subplot(1, 3, 3)
sns.histplot(df_rf['idade_cliente'], kde=True, color='purple')
plt.title(f'Imputação com Random Forest - idade_cliente')
plt.ylim(0, 200)  # Define os limites do eixo y

plt.tight_layout()
plt.show()

In [ ]:
# @title
# Preparando métodos avançados para MAR e MNAR
from sklearn.linear_model import LinearRegression

# Para tempo_cliente_dias (MAR) - Imputação baseada em regressão com idade
df_mar_regression = df.copy()
# Treinando modelo para prever tempo_cliente com base em idade
df_temp_mar = df[['idade_cliente', 'tempo_cliente_dias']].dropna()
if len(df_temp_mar) > 0:
    X_mar = df_temp_mar[['idade_cliente']]
    y_mar = df_temp_mar['tempo_cliente_dias']
    model_mar = LinearRegression()
    model_mar.fit(X_mar, y_mar)

    # Prevendo valores ausentes
    missing_tempo = df['tempo_cliente_dias'].isnull()
    if missing_tempo.sum() > 0:
        X_missing_tempo = df.loc[missing_tempo, ['idade_cliente']].dropna()
        if len(X_missing_tempo) > 0:
            predicted_tempo = model_mar.predict(X_missing_tempo)
            df_mar_regression.loc[X_missing_tempo.index, 'tempo_cliente_dias'] = predicted_tempo

# Para avaliacao_produto (MNAR) - Criando indicador + imputação condicional
df_mnar_indicator = df.copy()
# Criar indicador de ausência
df_mnar_indicator['avaliacao_missing'] = df_mnar_indicator['avaliacao_produto'].isnull().astype(int)
# Imputação condicional baseada em devolveu_produto
for devolveu in [0, 1]:
    mask = (df_mnar_indicator['devolveu_produto'] == devolveu) & df_mnar_indicator['avaliacao_produto'].isnull()
    if devolveu == 1:
        # Produtos devolvidos tendem a ter avaliações piores
        df_mnar_indicator.loc[mask, 'avaliacao_produto'] = 2.0
    else:
        # Produtos não devolvidos: usar a moda do grupo
        moda_grupo = df_mnar_indicator[df_mnar_indicator['devolveu_produto'] == 0]['avaliacao_produto'].mode()
        if len(moda_grupo) > 0:
            df_mnar_indicator.loc[mask, 'avaliacao_produto'] = moda_grupo[0]

# Comparando imputações para variáveis com padrões diferentes
plt.figure(figsize=(15, 10))

# Para idade_cliente (MCAR)
plt.subplot(2, 2, 1)
sns.kdeplot(df['idade_cliente'].dropna(), label='Original', fill=True, alpha=0.5)
sns.kdeplot(df_imputed['idade_cliente'], label='Média', fill=True, alpha=0.5)
sns.kdeplot(df_median['idade_cliente'], label='Mediana', fill=True, alpha=0.5)
sns.kdeplot(df_rf['idade_cliente'], label='Random Forest', fill=True, alpha=0.5)
plt.title('Comparação de Métodos - idade_cliente (MCAR)\nMétodos simples vs. avançados', fontsize=12)
plt.xlabel('Idade do Cliente')
plt.ylabel('Densidade')
plt.legend()

# Para tempo_cliente_dias (MAR)
plt.subplot(2, 2, 2)
sns.kdeplot(df['tempo_cliente_dias'].dropna(), label='Original', fill=True, alpha=0.5)
sns.kdeplot(df_imputed['tempo_cliente_dias'], label='Média', fill=True, alpha=0.5)
sns.kdeplot(df_median['tempo_cliente_dias'], label='Mediana', fill=True, alpha=0.5)
sns.kdeplot(df_mar_regression['tempo_cliente_dias'].dropna(), label='Regressão (idade)', fill=True, alpha=0.5, color='orange')
plt.title('Comparação de Métodos - tempo_cliente_dias (MAR)\nMétodos simples vs. regressão', fontsize=12)
plt.xlabel('Tempo Cliente (dias)')
plt.ylabel('Densidade')
plt.legend()

# Para avaliacao_produto (MNAR)
plt.subplot(2, 2, 3)
# Criando posições para as barras
x_pos = np.arange(1, 6)
width = 0.25

# Contando frequências
original_counts = df['avaliacao_produto'].dropna().value_counts().sort_index()
moda_counts = df_imputed['avaliacao_produto'].value_counts().sort_index()
indicator_counts = df_mnar_indicator['avaliacao_produto'].value_counts().sort_index()

# Garantindo que todas as avaliações estão representadas
for i in range(1, 6):
    if i not in original_counts.index:
        original_counts[i] = 0
    if i not in moda_counts.index:
        moda_counts[i] = 0
    if i not in indicator_counts.index:
        indicator_counts[i] = 0

original_counts = original_counts.sort_index()
moda_counts = moda_counts.sort_index()
indicator_counts = indicator_counts.sort_index()

# Plotando barras
plt.bar(x_pos - width, original_counts.values, width, label='Original', alpha=0.7, color='blue')
plt.bar(x_pos, moda_counts.values, width, label='Moda', alpha=0.7, color='red')
plt.bar(x_pos + width, indicator_counts.values, width, label='Indicador + Condicional', alpha=0.7, color='green')

plt.title('Comparação de Métodos - avaliacao_produto (MNAR)\nModa vs. Indicador + Imputação Condicional', fontsize=12)
plt.xlabel('Avaliação do Produto')
plt.ylabel('Frequência')
plt.xticks(x_pos)
plt.legend()

# Subplot 4: Resumo das estratégias
plt.subplot(2, 2, 4)
plt.axis('off')
strategies_text = """
 RESUMO DAS ESTRATÉGIAS APROPRIADAS:

 MCAR (idade_cliente):
   • Métodos simples funcionam bem
   • RF não traz grande vantagem

 MAR (tempo_cliente_dias):
   • Regressão preserva relações
   • Melhor que média/mediana

 MNAR (avaliacao_produto):
   • Indicador + condicional
   • Preserva padrão de ausência
   • Considera contexto (devolução)
"""
plt.text(0.1, 0.5, strategies_text, fontsize=14, verticalalignment='center')

plt.tight_layout()
plt.suptitle('Impacto de Diferentes Estratégias por Tipo de Ausência', fontsize=14, y=1.02)
plt.show()

# ==================================================
# 12. UTILIZANDO VARIÁVEIS INDICADORAS DE AUSÊNCIA
# ==================================================

In [ ]:
# @title
# Criando indicadores para valores ausentes
df_indicators = df.copy()

for col in numeric_cols_with_na:
    # Criando uma coluna indicadora (1 se ausente, 0 se presente)
    df_indicators[f'{col}_missing'] = df_indicators[col].isnull().astype(int)

    # Imputação com média para os valores ausentes
    df_indicators[col] = df_indicators[col].fillna(df_indicators[col].mean())

# Verificando as novas colunas indicadoras
print("\nPrimeiras linhas com indicadores de ausência:")
indicator_cols = [col for col in df_indicators.columns if '_missing' in col]
print(df_indicators[indicator_cols].head())

In [ ]:
# @title
# Analisando a utilidade dos indicadores
# Exemplo: Relação entre valores ausentes em avaliacao_produto e devolução
plt.figure(figsize=(10, 6))
sns.countplot(x='avaliacao_produto_missing', hue='devolveu_produto', data=df_indicators)
plt.title('Relação entre Ausência de Avaliação e Devolução de Produto')
plt.xlabel('Avaliação Ausente')
plt.ylabel('Contagem')
plt.xticks([0, 1], ['Avaliado', 'Não Avaliado'])
plt.legend(title='Devolveu Produto', labels=['Não', 'Sim'])
plt.show()

## 🎯 Micro-atividade 3: Pattern Hunter - Análise Guiada (10 min)

**Objetivo**: Usar uma função pronta para descobrir padrões de valores ausentes

**Cenário**: A diretoria suspeita que existem padrões sistemáticos nos dados ausentes. Você vai usar uma função de detecção automatizada.

### Parte 1: Função de detecção de padrões

In [ ]:
# @title
# 🔍 Função de Detecção de Padrões de Ausência

from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

def detectar_tipo_ausencia(df, var_alvo):
    """
    Detecta o tipo provável de ausência (MCAR, MAR, MNAR)
    para uma variável específica no DataFrame.
    """
    if df[var_alvo].isnull().sum() == 0:
        return "SEM_NA", 1.0, "Variável não possui valores ausentes"

    # Criando indicador de ausência
    df_temp = df.copy()
    df_temp[f'{var_alvo}_missing'] = df_temp[var_alvo].isnull()

    # Lista para armazenar p-valores de testes
    p_valores = []
    relacoes = []

    # Testando relação com outras variáveis (MAR)
    for col in df.columns:
        if col != var_alvo and df[col].dtype in ['float64', 'int64']:
            # Teste t para variáveis numéricas
            try:
                group_present = df_temp[df_temp[f'{var_alvo}_missing'] == False][col].dropna()
                group_missing = df_temp[df_temp[f'{var_alvo}_missing'] == True][col].dropna()

                if len(group_present) > 1 and len(group_missing) > 1:
                    t_stat, p_val = stats.ttest_ind(group_present, group_missing, equal_var=False)
                    p_valores.append(p_val)
                    if p_val < 0.05:
                        relacoes.append(col)
            except:
                continue

    # Análise de correlação com valor da própria variável (proxy para MNAR)
    # Verificamos se valores extremos têm mais chance de estar ausentes
    if var_alvo in ['avaliacao_produto', 'tempo_entrega_dias']:
        # Sabemos que estas estão relacionadas com devolveu_produto
        if 'devolveu_produto' in df.columns:
            corr_devolucao = df_temp[[f'{var_alvo}_missing', 'devolveu_produto']].corr().iloc[0, 1]
            if abs(corr_devolucao) > 0.3:
                return "MNAR", 0.001, f"Fortemente relacionado com devolução (corr={corr_devolucao:.3f})"

    # Decisão baseada nos testes
    if len(p_valores) == 0:
        return "MCAR", 0.99, "Sem relações significativas com outras variáveis"

    min_p = min(p_valores) if p_valores else 1.0

    if min_p < 0.01 and len(relacoes) > 0:
        return "MAR", min_p, f"Relacionado com: {', '.join(relacoes[:3])}"
    elif min_p < 0.05 and len(relacoes) > 0:
        return "MAR", min_p, f"Fracamente relacionado com: {relacoes[0]}"
    else:
        return "MCAR", min_p, "Padrão aleatório de ausência"

# Aplicando a função para todas as variáveis
print("🔍 RELATÓRIO DE DETECÇÃO DE PADRÕES DE AUSÊNCIA")
print("=" * 70)
print(f"{'Variável':<25} {'Tipo':<8} {'p-valor':<10} {'Explicação'}")
print("-" * 70)

resultados_deteccao = {}
for col in df.columns:
    if df[col].isnull().sum() > 0:
        tipo, p_val, explicacao = detectar_tipo_ausencia(df, col)
        resultados_deteccao[col] = {'tipo': tipo, 'p_valor': p_val, 'explicacao': explicacao}

        # Formatação com cores (simulada com símbolos)
        simbolo = "🔴" if tipo == "MNAR" else ("🟡" if tipo == "MAR" else "🟢")
        print(f"{simbolo} {col:<22} {tipo:<8} {p_val:<10.4f} {explicacao}")

print("=" * 70)
print("\n📊 Resumo:")
print(f"🟢 MCAR (aleatório): {sum(1 for r in resultados_deteccao.values() if r['tipo'] == 'MCAR')} variáveis")
print(f"🟡 MAR (relacionado): {sum(1 for r in resultados_deteccao.values() if r['tipo'] == 'MAR')} variáveis")
print(f"🔴 MNAR (não aleatório): {sum(1 for r in resultados_deteccao.values() if r['tipo'] == 'MNAR')} variáveis")

In [ ]:
# @title
# 💡 Estratégias Recomendadas para Cada Tipo

print("\n🎯 ESTRATÉGIAS DE IMPUTAÇÃO RECOMENDADAS:")
print("=" * 70)

estrategias = {
    'MCAR': {
        'metodos': ['Média', 'Mediana', 'Eliminação de linhas'],
        'justificativa': 'Dados aleatórios - métodos simples funcionam bem'
    },
    'MAR': {
        'metodos': ['KNN', 'Regressão', 'Random Forest'],
        'justificativa': 'Usar relações com outras variáveis para imputação'
    },
    'MNAR': {
        'metodos': ['Indicadores de ausência', 'Imputação múltipla', 'Modelos específicos'],
        'justificativa': 'Padrão sistemático requer tratamento especial'
    }
}

for col, info in resultados_deteccao.items():
    tipo = info['tipo']
    print(f"\n📌 {col} ({tipo}):")
    print(f"   Métodos sugeridos: {', '.join(estrategias[tipo]['metodos'])}")
    print(f"   Razão: {estrategias[tipo]['justificativa']}")

print("\n" + "=" * 70)

### Parte 2: Interpretação dos Resultados 🎯

**O que você descobriu?**

1. **Confirmação dos padrões**: A função detectou corretamente os tipos que criamos?
   - `idade_cliente`: MCAR ✅
   - `tempo_cliente_dias`: MAR ✅
   - `avaliacao_produto`: MNAR ✅

2. **Surpresas**: Alguma variável teve classificação inesperada?

3. **Implicações práticas**:
   - **MCAR**: Pode usar métodos simples sem muito prejuízo
   - **MAR**: DEVE usar informação de outras variáveis
   - **MNAR**: CUIDADO - pode introduzir viés significativo

**🏆 Você ganhou o título de "Data Detective"!**
Agora sabe identificar e tratar diferentes tipos de valores ausentes!

# ==================================================
# 13. RECOMENDAÇÕES E BOAS PRÁTICAS
# ==================================================

RECOMENDAÇÕES PARA LIDAR COM VALORES AUSENTES:

1. Entenda o contexto e tipo de ausência
   - Investigue por que os dados estão ausentes (MCAR, MAR, MNAR)
   - Envolva especialistas do domínio para compreender o significado

2. Documentação e transparência
   - Documente todas as decisões e métodos de tratamento
   - Seja transparente sobre as limitações das abordagens escolhidas

3. Prevenção é o melhor remédio
   - Melhore os processos de coleta de dados
   - Implemente validações e verificações de qualidade

4. Abordagem estratégica
   - Não aplique a mesma técnica para todas as variáveis
   - Considere o tipo de variável e padrão de ausência
   - Avalie o impacto de diferentes métodos

5. Sensibilidade e robustez
   - Realize análises de sensibilidade para métodos de imputação
   - Avalie como os resultados mudam com diferentes abordagens
   - Prefira métodos robustos para dados com muita incerteza

6. Fluxo de trabalho recomendado:
   1. Exploração e visualização de valores ausentes
   1. Identificação de padrões e tipos
   1. Decisão sobre remoção ou imputação
   1. Aplicação de técnicas apropriadas
   1. Avaliação do impacto das estratégias
   1. Documentação das escolhas e limitações